In [ ]:
import os, re, json
import shutil
import pandas as pd
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datetime import datetime

plt.style.use("default")
if shutil.which("latex"):
    plt.rc("text", usetex=True)
    plt.rc("font", family="serif", serif=["Computer Modern Roman"])

RESULTS_PATH = os.path.expanduser("~/ope/results")
FIGURES_DIR = os.path.expanduser("~/ope/results/figures/final")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Configuration ──────────────────────────────────────────────
datasets = ["mslr_pbm", "yahoo_pbm"]
datasets = ["mslr_ipm"]  # <-- for IPM experiment
bias_order = [
    "bias_pow0_8",
    "bias_plus0",
    "bias_pow1_2",
    "bias_minus0_1",
    "bias_zigzag0_1",
]
BIAS_LABELS = {
    "bias_pow0_6": r"$p^{0.6}$",
    "bias_pow0_8": r"$p^{0.8}$",
    "bias_plus0": r"$p^{1.0}$",
    "bias_pow1_2": r"$p^{1.2}$",
    "bias_pow1_4": r"$p^{1.4}$",
    "bias_minus0_1": r"$p-0.1$",
    "bias_zigzag0_1": "$p\pm z$",
}
fixed_temp, fixed_ns, fixed_pb, fixed_eps = 0.6, 5, "pbdcg", "eps_times1_5"
nsampled_values = [1, 5, 25]
exclude_models = {}  # {"learned_F_snips", "SLOPE", "w9_snips"}

# ── Helpers ────────────────────────────────────────────────────
FIXED_MODELS = [
    "learned_F",
    "w0",
    "w9",
    "SLOPE",
    "learned_F_snips",
    "w0_snips",
    "w9_snips",
    "SLOPE_SNIPS",
    "BLUE",
    "OPERA",
]
W_ALL = [f"w{i}" for i in range(10)]
W_ALL_SNIPS = [f"w{i}_snips" for i in range(10)]
_tab20 = plt.cm.tab20(range(20))
MODEL_COLORS = {
    "learned_F": _tab20[0],
    "w0": _tab20[5],
    "w9": _tab20[2],
    "w0_snips": _tab20[3],
    "SLOPE_SNIPS": _tab20[4],
    "BLUE": _tab20[1],
    "OPERA": _tab20[6],
    "SLOPE": _tab20[7],
    "learned_F_snips": _tab20[8],
    "w9_snips": _tab20[9],
    "best_w": _tab20[16],
    "best_w_snips": _tab20[17],
}
DISPLAY_NAMES = {
    "w0": "IPM",
    "w9": "PBM/SLOPE",
    "learned_F": "GPBM",
    "w0_snips": "SN-IPM",
    "SLOPE_SNIPS": "SN-INT",
    "best_w": r"INT$^*$",
    "best_w_snips": r"SN-INT$^*$",
    "learned_F_snips": "SN-GPBM",
    "w9_snips": "SN-PBM",
}


def parse_df(results_base, min_mtime=None):
    rows = []
    for root, dirs, files in os.walk(results_base):
        for f in files:
            if not f.endswith(".json"):
                continue
            path = os.path.join(root, f)
            if min_mtime and datetime.fromtimestamp(os.path.getmtime(path)) < min_mtime:
                continue
            with open(path) as fp:
                data = json.load(fp)
            rel = os.path.relpath(path, results_base)
            parts = rel.split("/")
            params, unnamed = {}, []
            for p in parts:
                if m := re.match(r"nsampled(\d+)", p):
                    params["nsampled"] = int(m.group(1))
                elif m := re.match(r"ndocs(\d+)", p):
                    params["ndocs"] = int(m.group(1))
                elif m := re.match(r"temp(.+)", p):
                    key = "temp1" if "temp1" not in params else "temp2"
                    params[key] = float(
                        m.group(1).replace("_neg", "-").replace("_", ".")
                    )
                elif p in ("pbdcg", "pbinvrank"):
                    params["pb"] = p
                elif p.startswith("bias_"):
                    params["bias"] = p
                elif p.startswith("eps_"):
                    params["eps"] = p
                elif p.startswith("fold"):
                    params["fold"] = int(p.split("fold")[-1])
                else:
                    unnamed.append(p)
            if len(unnamed) >= 3:
                params["dataset"], params["logging_model"], params["target_model"] = (
                    unnamed[:3]
                )
            for est_name, metrics in data.get("metrics", {}).items():
                if isinstance(metrics, dict):
                    row = {**params, "estimator": est_name}
                    row.update({f"metric_{k}": v for k, v in metrics.items()})
                    rows.append(row)
    return pd.DataFrame(rows)


def get_models_to_plot(grp):
    return {m: m for m in FIXED_MODELS if m in grp["estimator"].values}


def load_F_matrices(results_base):
    data = defaultdict(list)
    for root, dirs, files in os.walk(results_base):
        for f in files:
            if not f.startswith("repeat_") or not f.endswith(".pt"):
                continue
            path = os.path.join(root, f)
            rel = os.path.relpath(path, results_base)
            parts = rel.split("/")
            params = {}
            for p in parts:
                if m := re.match(r"nsampled(\d+)", p):
                    params["nsampled"] = int(m.group(1))
                elif m := re.match(r"temp(.+)", p):
                    key = "temp1" if "temp1" not in params else "temp2"
                    params[key] = float(
                        m.group(1).replace("_neg", "-").replace("_", ".")
                    )
                elif p in ("pbdcg", "pbinvrank"):
                    params["pb"] = p
                elif p.startswith("bias_"):
                    params["bias"] = p
                elif p.startswith("eps_"):
                    params["eps"] = p
                elif p.startswith("fold"):
                    params["fold"] = p
            if params.get("fold") != "fold1":
                continue
            key = (
                params.get("pb"),
                params.get("bias"),
                params.get("nsampled"),
                params.get("temp1"),
                params.get("temp2"),
                params.get("eps"),
            )
            F = torch.load(path, map_location="cpu")["F"]
            F /= F.sum(-1, keepdim=True)
            data[key].append(F.numpy() if isinstance(F, torch.Tensor) else F)
    return data

In [ ]:
# RQ1-2: MSE vs bias scatter
n_groups = len(bias_order)
group_spacing = 0.6


def compute_best_envelope_by_bias(df, estimators):
    w_data = df[df["estimator"].isin(estimators)]
    if len(w_data) == 0:
        return None
    best_per_fold = w_data.groupby(["bias", "fold"])["metric_mse"].min().reset_index()
    best = (
        best_per_fold.groupby("bias")["metric_mse"]
        .agg(["mean", "min", "max"])
        .reset_index()
    )
    best = best[best["bias"].isin(bias_order)]
    best["x"] = best["bias"].map(
        {b: j * group_spacing for j, b in enumerate(bias_order)}
    )
    return best.sort_values("x")


for i, dataset in enumerate(datasets):
    df_ds = parse_df(f"{RESULTS_PATH}/experiments/{dataset}")
    df_ds = df_ds[
        (df_ds["pb"] == fixed_pb)
        & (df_ds["temp1"] == fixed_temp)
        & (df_ds["nsampled"] == fixed_ns)
        & (df_ds["eps"] == fixed_eps)
    ]
    df_ds = df_ds[df_ds.bias.isin(bias_order)]
    if len(df_ds) == 0:
        print(f"No data for {dataset}")
        continue

    fig_w = 0.8 * n_groups
    fig_h = 1.1 if i == 0 else 1.25
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    models_map = {
        k: v for k, v in get_models_to_plot(df_ds).items() if k not in exclude_models
    }
    n_models = len(models_map) + 2
    idx = 0

    for est_name in models_map:
        est_data = df_ds[df_ds["estimator"] == est_name]
        if len(est_data) == 0:
            idx += 1
            continue
        agg = (
            est_data.groupby("bias")["metric_mse"]
            .agg(["mean", "min", "max"])
            .reset_index()
        )
        agg = agg[agg["bias"].isin(bias_order)]
        agg["x"] = agg["bias"].map(
            {b: j * group_spacing for j, b in enumerate(bias_order)}
        )
        agg = agg.sort_values("x")
        x_off = agg["x"] + (idx - n_models / 2 + 0.5) * 0.6 / n_models
        ax.errorbar(
            x_off,
            agg["mean"],
            yerr=[agg["mean"] - agg["min"], agg["max"] - agg["mean"]],
            fmt="o",
            color=MODEL_COLORS[est_name],
            capsize=1.5,
            capthick=0.5,
            markersize=2,
            linewidth=0.6,
        )
        idx += 1

    for label, estimators, marker in [
        ("best_w", W_ALL, "s"),
        ("best_w_snips", W_ALL_SNIPS, "s"),
    ]:
        best = compute_best_envelope_by_bias(df_ds, estimators)
        if best is not None:
            x_off = best["x"] + (idx - n_models / 2 + 0.5) * 0.6 / n_models
            ax.errorbar(
                x_off,
                best["mean"],
                yerr=[best["mean"] - best["min"], best["max"] - best["mean"]],
                fmt=marker,
                color=MODEL_COLORS[label],
                capsize=1.5,
                capthick=0.5,
                markersize=2,
                linewidth=0.6,
            )
        idx += 1

    for j in range(1, n_groups):
        ax.axvline(
            j * group_spacing - group_spacing / 2,
            color="gray",
            ls="--",
            alpha=0.3,
            lw=0.5,
        )
    ax.set_xticks([j * group_spacing for j in range(n_groups)])
    ax.set_xticklabels(
        [BIAS_LABELS[b] for b in bias_order] if i + 1 == len(datasets) else []
    )
    ax.set_yscale("log")
    ax.set_xlim(-group_spacing / 2, (n_groups - 1) * group_spacing + group_spacing / 2)
    ax.yaxis.set_minor_locator(
        matplotlib.ticker.LogLocator(base=10, subs=(1,), numticks=10)
    )
    ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    if "mslr" in dataset:
        ax.set_ylim(1.1e-6, 3e-3)
    else:
        ax.set_ylim(2e-6, 2.8e-2)

    plt.tight_layout()
    out_dir = f"{FIGURES_DIR}/{dataset}"
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(
        f"{out_dir}/{dataset}_mse_vs_bias_scatter_stacked_temp={fixed_temp}_ns={fixed_ns}.pgf",
        facecolor="white",
        dpi=500,
        bbox_inches="tight",
        pad_inches=0.05,
    )
    fig.savefig(
        f"{out_dir}/{dataset}_mse_vs_bias_scatter_stacked_temp={fixed_temp}_ns={fixed_ns}.pdf",
        facecolor="white",
        dpi=500,
        bbox_inches="tight",
        pad_inches=0.05,
    )
    plt.close(fig)
    print(f"Saved {out_dir}")

In [ ]:
# RQ1: MSE vs temp+nsampled
fixed_bias = "bias_pow1_2"


def compute_best_envelope_by_temp(df, estimators, temp_order):
    w_data = df[df["estimator"].isin(estimators)]
    if len(w_data) == 0:
        return None
    best_per_fold = w_data.groupby(["temp1", "fold"])["metric_mse"].min().reset_index()
    best = (
        best_per_fold.groupby("temp1")["metric_mse"]
        .agg(["mean", "min", "max"])
        .reset_index()
    )
    best = best[best["temp1"].isin(temp_order)]
    best["x"] = best["temp1"].map({t: i for i, t in enumerate(temp_order)})
    return best.sort_values("x")


for dataset in datasets:
    df_ds = parse_df(f"{RESULTS_PATH}/experiments/{dataset}")
    df_ds = df_ds[
        (df_ds["pb"] == fixed_pb)
        & (df_ds["bias"] == fixed_bias)
        & (df_ds["eps"] == fixed_eps)
        & (df_ds["temp1"] != -0.2)
    ]
    models_map = {
        k: v for k, v in get_models_to_plot(df_ds).items() if k not in exclude_models
    }
    n_models = len(models_map) + 2

    for ns in nsampled_values:
        df_ns = df_ds[df_ds["nsampled"] == ns]
        if len(df_ns) == 0:
            print(f"No data for {dataset} ns={ns}")
            continue

        temp_order = sorted(df_ns["temp1"].unique(), key=lambda t: 1 / t)
        temp_labels_map = {t: i for i, t in enumerate(temp_order)}
        n_groups = len(temp_order)

        plot_w, plot_h = 2, 1
        left_margin = 0.5 if ns == nsampled_values[0] else 0.1
        bottom_margin = 0.4
        fig_w = left_margin + plot_w + 0.1
        fig_h = bottom_margin + plot_h + 0.1
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.set_position(
            [left_margin / fig_w, bottom_margin / fig_h, plot_w / fig_w, plot_h / fig_h]
        )

        idx = 0
        for est_name in models_map:
            est_data = df_ns[df_ns["estimator"] == est_name]
            if len(est_data) == 0:
                idx += 1
                continue
            agg = (
                est_data.groupby("temp1")["metric_mse"]
                .agg(["mean", "min", "max"])
                .reset_index()
            )
            agg["x"] = agg["temp1"].map(temp_labels_map)
            agg = agg.sort_values("x")
            x_off = agg["x"] + (idx - n_models / 2 + 0.5) * 0.6 / n_models
            ax.errorbar(
                x_off,
                agg["mean"],
                yerr=[agg["mean"] - agg["min"], agg["max"] - agg["mean"]],
                fmt="o",
                color=MODEL_COLORS[est_name],
                capsize=1.5,
                capthick=0.5,
                markersize=2,
                linewidth=0.6,
            )
            idx += 1

        for label, estimators, marker in [
            ("best_w", W_ALL, "s"),
            ("best_w_snips", W_ALL_SNIPS, "s"),
        ]:
            best = compute_best_envelope_by_temp(df_ns, estimators, temp_order)
            if best is not None:
                x_off = best["x"] + (idx - n_models / 2 + 0.5) * 0.6 / n_models
                ax.errorbar(
                    x_off,
                    best["mean"],
                    yerr=[best["mean"] - best["min"], best["max"] - best["mean"]],
                    fmt=marker,
                    color=MODEL_COLORS[label],
                    capsize=1.5,
                    capthick=0.5,
                    markersize=2,
                    linewidth=0.6,
                )
            idx += 1

        for j in range(1, n_groups):
            ax.axvline(j - 0.5, color="gray", ls="--", alpha=0.3, lw=0.5)
        ax.set_xticks(range(n_groups))
        ax.set_xticklabels([rf"$\frac{{1}}{{{t}}}$" for t in temp_order])
        ax.set_yscale("log")
        ax.set_ylim(1e-6, 1e-2 if "mslr" in dataset else 3e-2)
        ax.set_xlim(-0.5, n_groups - 0.5)
        if ns != nsampled_values[0]:
            ax.yaxis.set_visible(False)
        if "mslr" in dataset:
            ax.xaxis.set_visible(False)

        out_dir = f"{FIGURES_DIR}/{dataset}"
        os.makedirs(out_dir, exist_ok=True)
        path = f"{out_dir}/{dataset}_mse_vs_temp_scatter_bias={fixed_bias}_ns={ns}.pgf"
        fig.savefig(path, facecolor="white")
        plt.close(fig)
        print(f"Saved {path}")

In [ ]:
# RQ1-2 Legend figure
for dataset in datasets:
    df_ds = parse_df(f"{RESULTS_PATH}/experiments/{dataset}")
    models_map = {
        k: v for k, v in get_models_to_plot(df_ds).items() if k not in exclude_models
    }
    models_map["best_w"] = "best_w"
    models_map["best_w_snips"] = "best_w_snips"

    fig, ax = plt.subplots(figsize=(0.1, 0.3))
    ax.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=MODEL_COLORS[k]) for k in models_map]
    labels = [DISPLAY_NAMES.get(k, v) for k, v in models_map.items()]
    ax.legend(
        handles,
        [rf"\textrm{{{l}}}" for l in labels],
        loc="center",
        ncol=len(models_map),
        frameon=False,
        handlelength=1,
        handletextpad=0.3,
        columnspacing=1,
    )

    out_dir = f"{FIGURES_DIR}/{dataset}"
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(
        f"{out_dir}/legend.pgf", facecolor="white", bbox_inches="tight", pad_inches=0
    )
    fig.savefig(
        f"{out_dir}/legend.pdf", facecolor="white", bbox_inches="tight", pad_inches=0
    )
    plt.close(fig)
    print(f"Saved {out_dir}/legend")

In [ ]:
# Heatmap: relative improvement of learned_F over best baseline across eps_k
heatmap_datasets = ["mslr_ipm"]
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

ALL_BASELINES = (
    W_ALL
    + W_ALL_SNIPS
    + [m for m in FIXED_MODELS if m != "learned_F" and m != "learned_F_snips"]
)
EPS_ORDER = [
    "eps_times0",
    "eps_times0_25",
    "eps_times0_5",
    "eps_times1",
    "eps_times1_5",
    "eps_times2",
    "eps_times3",
    "eps_times0_5_to_1_5",
    "eps_times1_5_to_0_5",
]
EPS_LABELS = {
    "eps_times0": r"$0$",
    "eps_times0_25": r"$0.25$",
    "eps_times0_5": r"$0.5$",
    "eps_times1": r"$1$",
    "eps_times1_5": r"$1.5$",
    "eps_times2": r"$2$",
    "eps_times3": r"$3$",
    "eps_times0_to_2": "$0{\\to}2$",
    "eps_times2_to_0": "$2{\\to}0$",
    "eps_times1_5_to_0_5": "$1.5{\\to}0.5$",
    "eps_times0_5_to_1_5": "$0.5{\\to}1.5$",
}
linthresh = 0.1


def symlog(x, lt=linthresh):
    return np.where(np.abs(x) <= lt, x, np.sign(x) * (lt + np.log1p(np.abs(x) - lt)))


# First pass: compute global vmin/vmax
global_vmin, global_vmax = np.inf, -np.inf
all_heatmap_data = {}
for dataset in heatmap_datasets:
    df_ds = parse_df(f"{RESULTS_PATH}/experiments/{dataset}")
    df_ds = df_ds[
        (df_ds["temp1"] > 0.2) & (df_ds["pb"] == "pbdcg") & (df_ds["nsampled"] >= 5)
    ]
    nsampled_vals = sorted(df_ds["nsampled"].dropna().unique())
    temp_vals = sorted(df_ds["temp1"].dropna().unique(), key=lambda t: 1 / t)

    improvements = {eps: [] for eps in EPS_ORDER}
    for bias in bias_order:
        for ns in nsampled_vals:
            for temp in temp_vals:
                subset = df_ds[
                    (df_ds["bias"] == bias)
                    & (df_ds["nsampled"] == ns)
                    & (df_ds["temp1"] == temp)
                ]
                baseline_data = subset[subset["estimator"].isin(ALL_BASELINES)]
                if len(baseline_data) == 0:
                    for eps in EPS_ORDER:
                        improvements[eps].append(np.nan)
                    continue
                best_baseline = baseline_data.groupby("fold")["metric_mse"].min().mean()
                for eps in EPS_ORDER:
                    lf_data = subset[
                        (subset["estimator"] == "learned_F") & (subset["eps"] == eps)
                    ]
                    if len(lf_data) == 0:
                        improvements[eps].append(np.nan)
                    else:
                        lf_mse = lf_data["metric_mse"].mean()
                        improvements[eps].append(
                            (best_baseline - lf_mse) / min(best_baseline, lf_mse)
                        )

    heatmap_data = np.array([improvements[eps] for eps in EPS_ORDER])
    all_heatmap_data[dataset] = (heatmap_data, nsampled_vals, temp_vals)
    global_vmin = min(global_vmin, np.nanmin(heatmap_data))
    global_vmax = max(global_vmax, np.nanmax(heatmap_data))

# Colormap
colors_neg = plt.cm.Blues_r(np.linspace(0.2, 1, 128))
colors_pos = plt.cm.Reds(np.linspace(0.2, 1, 128))
cmap = LinearSegmentedColormap.from_list(
    "blue_red", np.vstack([colors_neg, colors_pos])
)
vmin_d, vmax_d = symlog(global_vmin), symlog(global_vmax)
norm = TwoSlopeNorm(vcenter=0, vmin=vmin_d, vmax=vmax_d)

# Second pass: plot
for di, dataset in enumerate(heatmap_datasets):
    heatmap_data, nsampled_vals, temp_vals = all_heatmap_data[dataset]
    heatmap_display = symlog(heatmap_data)
    n_temps = len(temp_vals)
    n_ns = len(nsampled_vals)
    n_bias = len(bias_order)
    n_cols = n_bias * n_ns * n_temps
    show_labels = di == 0

    fig = plt.figure(figsize=(n_cols * 0.25, len(EPS_ORDER) * 0.4 + 1.5))
    ax = fig.add_axes([0.1, 0.05, 0.85, 0.65])

    sns.heatmap(
        heatmap_display,
        ax=ax,
        cmap=cmap,
        norm=norm,
        annot=False,
        square=True,
        xticklabels=False,
        yticklabels=[EPS_LABELS[e] for e in EPS_ORDER] if show_labels else False,
        cbar=None,
    )
    if show_labels:
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    # Hierarchical x-axis labels
    bias_lbls = [BIAS_LABELS[b] for b in bias_order]
    for b_idx, bias_lbl in enumerate(bias_lbls):
        x_start = b_idx * n_ns * n_temps
        x_center = x_start + n_ns * n_temps / 2
        ax.text(
            x_center,
            1.28,
            bias_lbl,
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            transform=ax.get_xaxis_transform(),
        )
        if b_idx > 0:
            ax.axvline(x_start, color="black", linewidth=1)

    for b_idx in range(n_bias):
        for ns_idx, ns in enumerate(nsampled_vals):
            x_start = b_idx * n_ns * n_temps + ns_idx * n_temps
            x_center = x_start + n_temps / 2
            ax.text(
                x_center,
                1.16,
                f"{int(ns)}",
                ha="center",
                va="bottom",
                fontsize=10,
                transform=ax.get_xaxis_transform(),
            )
            if ns_idx > 0:
                ax.axvline(x_start, color="gray", linewidth=0.5, linestyle="-")

    for i, temp in enumerate(temp_vals * n_ns * n_bias):
        ax.text(
            i + 0.5,
            1.03,
            rf"$\frac{{1}}{{{temp}}}$",
            ha="center",
            va="bottom",
            fontsize=10,
            transform=ax.get_xaxis_transform(),
        )

    for y in [1.01, 1.14, 1.255, 1.38]:
        ax.plot(
            [0, n_cols],
            [y, y],
            color="gray",
            linewidth=0.5,
            transform=ax.get_xaxis_transform(),
            clip_on=False,
        )
    if show_labels:
        for y, lbl in [(1.28, r"$\hat{p}$"), (1.16, r"$n$"), (1.03, r"$\tau$")]:
            ax.text(
                -0.5,
                y,
                lbl,
                ha="right",
                va="bottom",
                fontsize=12,
                transform=ax.get_xaxis_transform(),
                clip_on=False,
            )

    out_dir = f"{FIGURES_DIR}/{dataset}"
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(
        f"{out_dir}/{dataset}_heatmap_learned_F_improvement.pdf",
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.06,
    )
    plt.close(fig)

    # Standalone colorbar
    fig_cb, ax_cb = plt.subplots(figsize=(0.1, 2))
    gradient = np.linspace(vmin_d, vmax_d, 256).reshape(-1, 1)
    ax_cb.imshow(
        gradient,
        aspect="auto",
        cmap=cmap,
        norm=norm,
        origin="lower",
        extent=[0, 1, vmin_d, vmax_d],
    )
    ax_cb.set_xticks([])
    ax_cb.yaxis.tick_right()
    orig_ticks = [t for t in [-15, -5, -1, 0, 1, 5] if global_vmin <= t <= global_vmax]
    ax_cb.set_yticks([symlog(t) for t in orig_ticks])
    ax_cb.set_yticklabels([f"{t:.0f}" if abs(t) >= 1 else f"{t}" for t in orig_ticks])
    fig_cb.savefig(
        f"{out_dir}/{dataset}_heatmap_colorbar.pdf",
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.02,
    )
    plt.close(fig_cb)
    print(f"Saved {out_dir}/{dataset}_heatmap_learned_F_improvement.pdf")

In [ ]:
# RQ4 heatmaps
plt.rc("axes", labelcolor="black", edgecolor="black")
plt.rc("xtick", color="black")
plt.rc("ytick", color="black")

include_combos = [
    # (bias, temp, ns)
    ("bias_minus0_1", 1.0, 1),
    ("bias_minus0_1", 1.0, 25),
    ("bias_minus0_1", 0.6, 1),
    ("bias_minus0_1", 0.6, 25),
    ("bias_zigzag0_1", 1.0, 1),
    ("bias_zigzag0_1", 1.0, 25),
]
temp2_fixed = 1.0
vmin, vmax = 0.1, 0.99

# Colorbar
fig_cb, ax_cb = plt.subplots(figsize=(0.3, 4))
sm = plt.cm.ScalarMappable(cmap="turbo", norm=plt.Normalize(vmin, vmax))
plt.colorbar(sm, cax=ax_cb)
fig_cb.savefig(f"{FIGURES_DIR}/F_colorbar.pdf", bbox_inches="tight", facecolor="white")
plt.close(fig_cb)

for dataset in datasets:
    F_data = load_F_matrices(f"{RESULTS_PATH}/experiments/{dataset}")
    eps_values_F = sorted(set(k[5] for k in F_data if k[5]))
    pb_values_F = sorted(set(k[0] for k in F_data if k[0]))

    for bias, t, ns in include_combos:
        for eps in eps_values_F:
            for pb in pb_values_F:
                key = (pb, bias, ns, t, temp2_fixed, eps)
                if key not in F_data or len(F_data[key]) == 0:
                    continue
                fig, ax = plt.subplots(figsize=(4, 4))
                F = np.stack(F_data[key])
                F_avg = np.mean(F / F.max(-1, keepdims=True), axis=0)
                ax.grid(False)
                ax.set_xticks([i - 0.5 for i in range(1, 10)], minor=True)
                ax.set_yticks([i - 0.5 for i in range(1, 10)], minor=True)
                ax.grid(which="minor", color="grey", linewidth=0.1, alpha=0.2)
                ax.tick_params(which="minor", length=0)
                ax.tick_params(which="major", labelsize=12)
                ax.imshow(F_avg, cmap="turbo", aspect="auto", vmin=vmin, vmax=vmax)
                ax.set_xticks(range(10))
                ax.set_yticks(range(10))
                ax.set_xticklabels([f"${i}$" for i in range(1, 11)])
                ax.set_yticklabels([f"${i}$" for i in range(1, 11)] if t == 0.6 else [])
                out_dir = f"{FIGURES_DIR}/{dataset}"
                os.makedirs(out_dir, exist_ok=True)
                fig.savefig(
                    f"{out_dir}/F_{pb}_{bias}_ns{ns}_t{t}_{eps}.pdf",
                    dpi=300,
                    bbox_inches="tight",
                    facecolor="white",
                    pad_inches=0,
                )
                plt.close(fig)
                print(f"Saved F_{pb}_{bias}_ns{ns}_t{t}_{eps} for {dataset}")